In [ ]:
import os
import re
import time
import asyncio
import urllib.parse
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from playwright.async_api import async_playwright

load_dotenv(Path("../../.env"))

PROJECT_ROOT = Path("../..").resolve()
STORE_DIR    = PROJECT_ROOT / "data" / "raw_data" / "store_info"
OUT_DIR      = PROJECT_ROOT / "ai" / "outputs"
OUT_PATH     = OUT_DIR / "naver_place_info.csv"

FOOD_CATS = [
    '한식', '분식', '중식', '일식', '양식', '경양식',
    '패밀리레스토랑', '비알코올', '주점', '카페', '기타 간이', '호프/통닭'
]

UA = ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
      "AppleWebKit/537.36 (KHTML, like Gecko) "
      "Chrome/120.0.0.0 Safari/537.36")

# place_id 추출 패턴 (map.naver.com/p/search/.../place/<id> 형태)
PLACE_ID_RE = re.compile(r'map\.naver\.com/p/(?:search/[^/"]+/place|entry/place)/(\d+)')
PRICE_RE    = re.compile(r'([\d,]{3,8})(?=원)')

VISIT_KEYWORDS = [
    "혼밥", "혼술", "혼자",
    "데이트", "커플", "기념일",
    "가족", "아이", "어린이",
    "회식", "단체", "접대",
    "친구", "모임", "브런치"
]

print("설정 완료")

In [ ]:
latest_dir = sorted(STORE_DIR.iterdir())[-1]
print(f"최신 분기: {latest_dir.name}")

df_store = pd.read_csv(latest_dir / "상가_서울.csv", encoding="utf-8-sig", low_memory=False)
df_store = df_store[df_store["상권업종중분류명"].isin(FOOD_CATS)].copy()
df_store = df_store[
    ["상가업소번호", "상호명", "상권업종중분류명", "행정동명", "도로명주소", "경도", "위도"]
].reset_index(drop=True)

if OUT_PATH.exists():
    done = set(pd.read_csv(OUT_PATH, encoding="utf-8-sig")["상가업소번호"].tolist())
    df_store = df_store[~df_store["상가업소번호"].isin(done)]
    print(f"이미 수집: {len(done):,}개 / 남은 대상: {len(df_store):,}개")
else:
    print(f"수집 대상: {len(df_store):,}개")

print(df_store.head(3).to_string())

In [ ]:
async def find_place_id(page, store_name: str, address: str) -> str | None:
    """
    네이버 검색 결과 페이지 HTML에서 place_id 추출.
    map.naver.com/p/.../place/<id> 패턴 사용.
    """
    addr_short = " ".join((address or "").split()[1:3])  # '강남구 역삼동'
    query = f"{store_name} {addr_short}".strip()
    enc   = urllib.parse.quote(query)

    try:
        await page.goto(
            f"https://search.naver.com/search.naver?query={enc}&where=place",
            wait_until="networkidle", timeout=15000
        )
        await page.wait_for_timeout(800)
        html = await page.content()
        ids  = PLACE_ID_RE.findall(html)
        return ids[0] if ids else None
    except:
        return None


PURPOSE_MAP = {
    "1인":      ["혼밥", "혼술", "혼자"],
    "커플":     ["데이트", "커플", "기념일"],
    "가족":     ["가족", "아이", "어린이"],
    "회식/접대": ["회식", "단체", "접대"],
    "친목":     ["친구", "모임", "브런치"],
}

def classify_visit_purpose(tags_str: str) -> str:
    purposes = [p for p, kws in PURPOSE_MAP.items() if any(k in tags_str for k in kws)]
    return "|".join(purposes) if purposes else "기타"

def classify_price_tier(avg) -> str:
    if avg is None:    return "알수없음"
    if avg < 10_000:   return "저가"
    if avg < 30_000:   return "중가"
    return "고가"


async def scrape_place(page, place_id: str) -> dict:
    """메뉴 가격 + 방문자 태그 수집 (pcmap.place.naver.com 사용)"""
    out = {"avg_price": None, "price_min": None, "price_max": None,
           "menu_count": 0, "visitor_tags": ""}

    # ── 메뉴 가격 (/menu 탭) ─────────────────────────────────────
    try:
        await page.goto(
            f"https://pcmap.place.naver.com/restaurant/{place_id}/menu",
            wait_until="domcontentloaded", timeout=12000
        )
        await page.wait_for_timeout(1500)
        html = await page.content()
        prices = []
        for m in PRICE_RE.findall(html):
            v = int(m.replace(",", ""))
            if 1_000 <= v <= 500_000:
                prices.append(v)
        if prices:
            out.update({
                "avg_price":  round(sum(prices) / len(prices)),
                "price_min":  min(prices),
                "price_max":  max(prices),
                "menu_count": len(prices),
            })
    except:
        pass

    # ── 방문자 태그 (/review/visitor 탭) ────────────────────────────
    # inner_text()로 실제 렌더링된 텍스트만 추출 → UI 템플릿 키워드 오탐 방지
    try:
        await page.goto(
            f"https://pcmap.place.naver.com/restaurant/{place_id}/review/visitor",
            wait_until="domcontentloaded", timeout=12000
        )
        await page.wait_for_timeout(1500)
        text = await page.inner_text("body")
        out["visitor_tags"] = "|".join(kw for kw in VISIT_KEYWORDS if kw in text)
    except:
        pass

    return out


print("함수 정의 완료")

In [ ]:
# ── 5개 상가 테스트 ───────────────────────────────────────────────
async def run_test():
    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=True)
        page = await browser.new_page(user_agent=UA)

        for i in range(5):
            row = df_store.iloc[i]
            pid = await find_place_id(page, row["상호명"], row["도로명주소"])
            if pid:
                info = await scrape_place(page, pid)
                avg  = info["avg_price"]
                price_str = f"{avg:,}원" if avg else "없음"
                print(f"[OK] {row['상호명']} ({row['상권업종중분류명']})")
                print(f"     place_id={pid}")
                print(f"     평균가격={price_str} [{classify_price_tier(avg)}]")
                print(f"     방문태그={info['visitor_tags'] or '없음'}")
                print(f"     방문목적={classify_visit_purpose(info['visitor_tags'])}")
            else:
                print(f"[--] {row['상호명']} — place_id 매칭 실패")
            await asyncio.sleep(1.2)

        await browser.close()

await run_test()

In [ ]:
# ── 전체 크롤링 (이어받기 가능) ───────────────────────────────────
BATCH_SIZE = 100
DELAY_SEC  = 1.2

async def run_crawl():
    results = []
    errors  = 0

    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=True)
        page = await browser.new_page(user_agent=UA)

        total = len(df_store)
        for seq, (_, row) in enumerate(df_store.iterrows(), 1):
            try:
                pid  = await find_place_id(page, row["상호명"], row["도로명주소"])
                info = await scrape_place(page, pid) if pid else \
                       {"avg_price": None, "price_min": None, "price_max": None,
                        "menu_count": 0, "visitor_tags": ""}

                results.append({
                    "상가업소번호":      row["상가업소번호"],
                    "상호명":           row["상호명"],
                    "상권업종중분류명":  row["상권업종중분류명"],
                    "행정동명":          row["행정동명"],
                    "도로명주소":        row["도로명주소"],
                    "경도":             row["경도"],
                    "위도":             row["위도"],
                    "place_id":         pid,
                    "avg_price":        info["avg_price"],
                    "price_min":        info["price_min"],
                    "price_max":        info["price_max"],
                    "menu_count":       info["menu_count"],
                    "visitor_tags":     info["visitor_tags"],
                    "price_tier":       classify_price_tier(info["avg_price"]),
                    "visit_purpose":    classify_visit_purpose(info["visitor_tags"]),
                })

            except Exception as e:
                errors += 1
                results.append({"상가업소번호": row["상가업소번호"], "상호명": row["상호명"],
                                "price_tier": "오류", "visit_purpose": "오류"})

            if seq % 10 == 0:
                matched = sum(1 for r in results if r.get("place_id"))
                priced  = sum(1 for r in results if r.get("avg_price"))
                print(f"[{seq:,}/{total:,}] 매칭:{matched/seq:.0%} 가격:{priced/seq:.0%} 오류:{errors}")

            if seq % BATCH_SIZE == 0:
                _df = pd.DataFrame(results)
                mode, header = ("a", False) if OUT_PATH.exists() else ("w", True)
                _df.to_csv(OUT_PATH, mode=mode, header=header, index=False, encoding="utf-8-sig")
                results = []
                print(f"  → {seq:,}개 저장")

            await asyncio.sleep(DELAY_SEC)

        await browser.close()

    if results:
        _df = pd.DataFrame(results)
        mode, header = ("a", False) if OUT_PATH.exists() else ("w", True)
        _df.to_csv(OUT_PATH, mode=mode, header=header, index=False, encoding="utf-8-sig")

    print(f"\n완료! → {OUT_PATH}")

await run_crawl()

In [ ]:
# ── 결과 요약 ─────────────────────────────────────────────────────
df_r  = pd.read_csv(OUT_PATH, encoding="utf-8-sig")
total = len(df_r)

print(f"전체:           {total:,}개")
print(f"place_id 매칭:  {df_r['place_id'].notna().sum():,}개 ({df_r['place_id'].notna().mean():.1%})")
print(f"가격 정보 확보: {df_r['avg_price'].notna().sum():,}개 ({df_r['avg_price'].notna().mean():.1%})")

print("\n[가격 티어 분포]")
print(df_r["price_tier"].value_counts())

print("\n[방문 목적 TOP 10]")
print(df_r["visit_purpose"].value_counts().head(10))

print("\n[업종별 평균 객단가]")
print(
    df_r.groupby("상권업종중분류명")["avg_price"].mean()
    .dropna().sort_values(ascending=False)
    .apply(lambda x: f"{x:,.0f}원")
)

print("\n[샘플 10개]")
print(df_r[["상호명","행정동명","avg_price","price_tier","visit_purpose"]]
      .dropna(subset=["avg_price"]).head(10).to_string())